In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset
import pickle as pl
import json
import re
import torch

/home/aditya/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import json
import torch
import pickle
import argparse
import re
from pathlib import Path
from datasets import load_dataset,Dataset
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification, BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

2026-04-22 06:49:16.638089: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-22 06:49:16.656469: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-22 06:49:16.656487: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-22 06:49:16.657034: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-22 06:49:16.660522: I tensorflow/core/platform/cpu_feature_guar

In [3]:
model = 'gpt2'
dataset = "ag_news"
base_dir = f"results_datainfo/{model}/{dataset}"
total_points = 50000
output_dir = f"./results_losses/{model}/{dataset}"
num_classes = 4

In [4]:
def return_data(dataset_name):
    """
    Load dataset from HuggingFace, handling special case for Yelp reviews.
    
    Args:
        dataset_name: Name of the dataset to load
        
    Returns:
        Loaded HuggingFace dataset object
    """
    # Yelp reviews require loading from 'yelp_review_full' repo
    if dataset_name == "yelp_review":
        return load_dataset("yelp_review_full")
    
    # All other datasets use their standard names
    return load_dataset(dataset_name)

In [ ]:
"""
Compute per-sample losses for base and expert models across all experiments.

For each experiment:
1. Load finetuned indices from dataset_info.json
2. Sample non-finetuned points to reach k total samples
3. Compute losses via forward pass
4. Save losses.pkl with indices, labels, and per-sample losses
"""

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]['text']
        label = self.data[idx]['label']
        
        encoding = self.tokenizer(
            text, max_length=self.max_length, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


def compute_losses(exp_dir, train_data, tokenizer, k_samples, device,num_classes):
    """Compute per-sample losses for one experiment."""
    
    # print(tokenizer.pad_token)
    # print(tokenizer.eos_token)
    
    # Load finetuned indices
    with open(exp_dir / 'dataset_info.json', 'r') as f:
        dataset_info = json.load(f)
    
    finetuned_indices = set(dataset_info['indices_D_prime'])
    n_finetuned = len(finetuned_indices)
    
    # Sample non-finetuned indices
    all_indices = set(range(len(train_data)))
    
    valid_indices = []
    for idx in all_indices:
        if train_data[idx]['label'] >=0:
            valid_indices.append(idx)
    valid_indices = set(valid_indices)
    
    non_finetuned_indices = list(valid_indices - finetuned_indices)
    n_non_finetuned = min(k_samples - n_finetuned, len(non_finetuned_indices))
    sampled_non_finetuned = random.sample(non_finetuned_indices, n_non_finetuned)
    
    # Combine and create dataset
    eval_indices = list(finetuned_indices) + sampled_non_finetuned
    is_finetuned = [True] * n_finetuned + [False] * n_non_finetuned
    
    eval_data = train_data.select(eval_indices)
    dataset = SimpleDataset(eval_data, tokenizer)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    
    # Determine model architecture
    if model == 'gpt2':
        base_model = GPT2ForSequenceClassification.from_pretrained('openai-community/gpt2', num_labels=num_classes)
        expert_model = GPT2ForSequenceClassification.from_pretrained('openai-community/gpt2', num_labels=num_classes)
    else:  # bert
        base_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_classes)
        expert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_classes)

    # Load saved state dicts
    base_model.load_state_dict(torch.load(exp_dir / 'theta_base_model.pt', map_location=device))
    expert_model.load_state_dict(torch.load(exp_dir / 'theta_exp_model.pt', map_location=device))

    base_model.to(device).eval()
    expert_model.to(device).eval()
    
    base_model.config.pad_token_id = tokenizer.pad_token_id
    expert_model.config.pad_token_id = tokenizer.pad_token_id
    
    # Compute losses
    base_losses, expert_losses = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Computing losses"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Base model
            base_outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            base_loss_per_sample = torch.nn.functional.cross_entropy(
                base_outputs.logits, labels, reduction='none'
            )
            base_losses.extend(base_loss_per_sample.cpu().numpy())
            
            # Expert model
            expert_outputs = expert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            expert_loss_per_sample = torch.nn.functional.cross_entropy(
                expert_outputs.logits, labels, reduction='none'
            )
            expert_losses.extend(expert_loss_per_sample.cpu().numpy())
    
    print(f"  ✓ Processed {len(eval_indices)} samples (finetuned: {n_finetuned}, non-finetuned: {n_non_finetuned})")
    
    labels = [train_data[idx]['label'] for idx in eval_indices]
    match = re.search(r'\(([\d., ]+)\)', exp_dir.name)
    
    return {
        'indices': eval_indices,
        'is_finetuned': is_finetuned,
        'base_losses': base_losses,
        'expert_losses': expert_losses,
        'labels': labels,
        'true_proportion':  np.array([float(x) for x in match.group(1).split(',')])
    }


def main():
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load tokenizer
    if model == 'gpt2':
        tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2')
        # the tokenizer masks the eos token by default , so we are assinging the pad token as the eos token
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    # Load dataset
    data =  return_data(dataset)
    train_data = data['train']
    print(f"Loaded {dataset}: {len(train_data)} samples\n")
    
    # Process all experiments
    base_path = Path(base_dir)
    exp_dirs = [d for d in base_path.iterdir() if d.is_dir() and (d / 'dataset_info.json').exists()]
    print(f"Found {len(exp_dirs)} experiment directories\n")
    
    
    for exp_dir in exp_dirs:
        print(f"Processing: {exp_dir.name}")
        
        losses_data = compute_losses(exp_dir, train_data, tokenizer, total_points, device, num_classes)
        print(exp_dir)
        
        output_root = Path(output_dir) / exp_dir.name 
        output_root.mkdir(parents=True, exist_ok=True)
        output_file =  f"{output_root}/losses.pkl"
        with open(output_file, 'wb') as f:
            pickle.dump(losses_data, f)
        
        print(f"  ✓ Saved to {output_file}\n")
    
    print("✓ All experiments processed")


if __name__ == '__main__':
    main()

Using device: cuda
Loaded ag_news: 120000 samples

Found 37 experiment directories

Processing: ag_news_(0.303, 0.019, 0.339, 0.339)


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  Computing losses:   2%|█▊                                                                                | 34/1563 [00:16<12:41,  2.01it/s]